## Execução local
Selecione o kernel **Python (FASE7)** e execute em ordem. Consulte `README_LOCAL.md` para instalação. Redes neurais usam PyTorch/CUDA; operações NumPy, áudio e modelos scikit-learn usam CPU.

In [ ]:
from pathlib import Path
import os, sys
# Localiza FASE7 tanto a partir do projeto quanto da pasta do notebook.
base = Path.cwd().resolve()
FASE7 = next((p for a in (base, *base.parents) for p in (a, a / "FASE7", a / "ANO1" / "FASE7") if p.name == "FASE7" and (p / "RNN").is_dir()), None)
if FASE7 is None:
    raise FileNotFoundError("Abra o Jupyter dentro da pasta FIAP ou FASE7.")
sys.path.insert(0, str(FASE7))
os.chdir(FASE7 / 'Cap 02')
print("Pasta de trabalho:", Path.cwd())
from fase7_torch import DEVICE
print("Dispositivo:", DEVICE)


FIAP

# TIAO - Redes Neurais

# Aula 4a - Predição de série temporal com LSTM

Predição de ações da TATA Global. Dados disponíveis em: https://www.kaggle.com/datasets/hiimanshuagarwal/nse-tataglobal


In [ ]:
# Import
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
dataset_train = pd.read_csv("LSTM_TATA_train.csv").sort_values("Date").reset_index(drop=True)
dataset_train

Nossa predição será no preço de abertura da bolsa, sempre nesta coluna. Então vamos pegar a coluna 1 do dataframe, puxando apenas seus valores (.values) e não a Série indexada o Pandas.

In [ ]:
training_set = dataset_train.iloc[:, 1:2].values
training_set

In [ ]:
# Plotando a série histórica
plt.figure(figsize=(20, 9))
plt.plot(range(dataset_train.shape[0]),
         dataset_train["Open"])
plt.title("Série temporal da ação")
plt.xticks(range(0, dataset_train["Date"].shape[0], 50),
           dataset_train["Date"][::50],
           rotation=45)
plt.xlabel("Datas")
plt.ylabel("Preço de abertura")
plt.show()


In [ ]:
# Normalizar os dados
from sklearn.preprocessing import MinMaxScaler

sc = MinMaxScaler(feature_range=(0, 1))
training_set_scaled = sc.fit_transform(training_set)

A parte a seguir é a mais importante! Como vamos criar um dataset para uso de **séries temporais**? Precisamos considerar o *tempo* na análise! Então, cada entrada do dataset deve conter uma série de valores anteriores. Aqui usamos uma janela de 60 dias anteriores, mas vocês podem analisar os resultados com janelas maiores ou menores.

Então, para cada dia, o próprio valor vira o label, e os dados que o predizem são cópias dos 60 valores anteriores.

In [ ]:
# Criar o dataset para série temporal
X_train = []
y_train = []
for i in range(60, len(training_set_scaled)):
  X_train.append(training_set_scaled[i-60:i, 0])
  y_train.append(training_set_scaled[i, 0])

X_train, y_train = np.array(X_train), np.array(y_train)

In [ ]:
X_train.shape

In [ ]:
X_train = np.reshape(X_train, (X_train.shape[0], # Número de exemplos de séries temporais
                               X_train.shape[1], # A série temporal deste exemplo (número de features)
                               1))               # Número de dimensões dos dados (1)

In [ ]:
X_train.shape

In [ ]:
from fase7_torch import RegressorLSTM, treinar, prever
import torch
from torch import nn

In [ ]:
regressor = RegressorLSTM(1, hidden=50, layers=4, dropout=0.2).to(DEVICE)
regressor

In [ ]:
optimizer = torch.optim.Adam(regressor.parameters())
criterion = nn.MSELoss()

In [ ]:
historico = treinar(regressor, X_train, y_train, optimizer, criterion, epochs=100, batch_size=32)

Para avaliar nosso modelo vamos usar como teste os dias seguintes aos dados do treino. Neste dataset temos as operações da bolsa do mês de outubro (no treinamento tivemos de meses anteriores até setembro).

Novamente vamos usar apenas a coluna do valor da ação na abertura da bolsa (coluna 1).

In [ ]:
dataset_test = pd.read_csv("LSTM_TATA_test.csv").sort_values("Date").reset_index(drop=True)
real_stock_price = dataset_test.iloc[:, 1:2].values

In [ ]:
dataset_test

Para compor o conjunto de teste de modo adequado, precisamos notar que o label do dia 1/out é o valor da ação naquele dia, e seus dados de treino são os de 60 dias anteriores (agosto e setembro). Para prever o valor da bolsa no dia 2/out, o valor real é o label, e os dados são os 60 valores anteriores, partindo do dia 1/out para trás. Seguindo esse raciocínio dia a dia, no último dia do mês (31/out), o valor real será o label, e os dados serão um vetor de tamanho 60, listando os valores anteriores de 30/out até 1/set.

Por isso, começamos concatenando os valores do treino e do teste para termos todos os dias de dados, e na sequêndia normalizamos os dados como o mesmo escalador que definimos anteriormente.

Agora, usamos os dados normalizados para aplicar o mesmo procedimento de construção do dataset com série histórica.

In [ ]:
dataset_total = pd.concat((dataset_train["Open"],
                           dataset_test["Open"]),
                          axis=0)
inputs = dataset_total[(len(dataset_total) - len(dataset_test)) - 60:].values
inputs = inputs.reshape(-1, 1) # "-1" infere o valor a partir dos dados
inputs = sc.transform(inputs)

X_test = []
for i in range(60, 60 + len(dataset_test)):
  X_test.append(inputs[i-60:i, 0])

X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0],
                             X_test.shape[1],
                             1))

In [ ]:
X_test.shape

In [ ]:
# Prevendo o valor do teste
predicted_stock_price = prever(regressor, X_test)
predicted_stock_price = sc.inverse_transform(predicted_stock_price)

In [ ]:
# Plotar o gráfico do resultado
plt.plot(real_stock_price,
         color="black",
         label="Ações da TATA")
plt.plot(predicted_stock_price,
         color="green",
         label="Ações PREDITAS da TATA")
plt.title("Predições da ações da TATA Consultoria")
plt.xticks(range(len(dataset_test)),
           dataset_test["Date"],
           rotation=45)
plt.xlabel("Tempo")
plt.ylabel("Preço da ação")
plt.legend()
plt.show()

In [ ]:
dataset_test["Date"].shape[0]